## Introducción

Los modelos de aprendizaje supervisado se basan en algoritmos que esperan datos previamente etiquetados, de manera que el algoritmo pueda validar durante la etapa de entrenamiento qué tan bien se desempeña. Esta etiqueta (`label`) puede ser una variable discreta, categórica o continua. Dependiendo de esto, sabremos si estamos ante un problema de clasificación o uno de regresión.

Entre los algoritmos más representativos de aprendizaje supervisado se encuentran: regresión lineal, regresión logística, Naive Bayes, árboles de decisión, bosque aleatorio (*random forest*), *gradient boosted trees*, kNN y SVM lineal. La librería MLlib de Spark cuenta con API para la mayoría de estos: `LinearRegression`, `LogisticRegression`, `DecisionTreeClassifier` / `DecisionTreeRegressor`, `RandomForestClassifier` / `RandomForestRegressor`, `GBTClassifier` / `GBTRegressor`, `NaiveBayes` y `LinearSVC`.

### Referencias:

Polak, A. (2023). Scaling machine learning with Spark. O'Reilly Media. Consultado en: https://learning.oreilly.com/library/view/scaling-machine-learning/9781098106812/ch06.html#supervised_machine_learning

## Inicialización y carga de datos

### Inicia ambiente

In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark


In [7]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Crea sesión Spark
spark = SparkSession.builder \
    .appName("nyc_taxi_eda") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

### Descarga datos de kaggle

In [3]:
import kagglehub
file_path = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data")

print("Path to dataset files:", file_path)

Using Colab cache for faster access to the 'nyc-yellow-taxi-trip-data' dataset.
Path to dataset files: /kaggle/input/nyc-yellow-taxi-trip-data


### Definir esquema

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, FloatType, BooleanType, DoubleType

# Define the schema
schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("RateCodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", BooleanType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("extra", FloatType(), True),
    StructField("mta_tax", FloatType(), True),
    StructField("tip_amount", FloatType(), True),
    StructField("tolls_amount", FloatType(), True),
    StructField("improvement_surcharge", FloatType(), True),
    StructField("total_amount", FloatType(), True)
])

# Spark reads the 4 CSVs with the predefined schema
df = spark.read.csv(
    file_path + "/*.csv",
    header=True,
    schema=schema, # Use the defined schema
    timestampFormat="yyyy-MM-dd HH:mm:ss"
)

## Selección de datos

Para este ejercicio, nos quedaremos con las variables de geoespaciales, además de tiempo de recogida y llegada. Estas son las utilizadas para las particiones realizadas en el parte 2 del proyecto.

El objetivo será predecir cuánto tiempo tomará un viaje dependiendo el punto de inicio y de llegada, así como la hora y día de inicio. Las variables de código y cantidad de la tarifa no nos sirven para este objetivo e incluso podrían provocar un filtrado de datos en el modelo, ya que la tarifa no solo depende del tiempo, sino también de la distancia.

In [9]:
df = df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude"
)


### Limpieza de datos

In [10]:
df = df.filter(
    # Filtrado geográfico: bounding box de NYC (elimina (0,0) y fuera de área)
    F.col("pickup_latitude").between(40.5, 41.0) &
    F.col("pickup_longitude").between(-74.3, -73.7)
)

print("Crear cache")

# Cache: D limpia se reutiliza en todas las particiones
df = df.cache()

n_limpio = df.count()
print(f"Filas tras limpieza (D limpia) : {n_limpio:,}")


Crear cache
Filas tras limpieza (D limpia) : 46,468,938


### Creación de variables de caracterización

In [11]:
df = df.withColumn("pickup_weekday", F.dayofweek("tpep_pickup_datetime"))
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))


# Variable A: franja_horaria
df = df.withColumn(
    "franja_horaria",
    F.when(F.col("pickup_hour").between(0, 5),   "madrugada")
     .when(F.col("pickup_hour").between(6, 11),  "manana")
     .when(F.col("pickup_hour").between(12, 17), "tarde")
     .otherwise("noche")
)

# Variable B: zona_geo
df = df.withColumn(
    "zona_geo",
    F.when(
        F.col("pickup_latitude").between(40.700, 40.878) &
        F.col("pickup_longitude").between(-74.020, -73.907),
        "manhattan"
    ).when(
        F.col("pickup_latitude").between(40.620, 40.665) &
        F.col("pickup_longitude").between(-73.815, -73.748),
        "jfk"
    ).when(
        F.col("pickup_latitude").between(40.760, 40.795) &
        F.col("pickup_longitude").between(-73.895, -73.855),
        "laguardia"
    ).otherwise("outer")
)

# Variable C: tipo_dia  (dayofweek: 1=Dom, 2=Lun, 3=Mar, 4=Mie, 5=Jue, 6=Vie, 7=Sab)
df = df.withColumn(
    "tipo_dia",
    F.when(F.col("pickup_weekday").isin([6, 7]), "fin_de_semana")   # Vie, Sab
     .when(F.col("pickup_weekday") == 1,          "valle")           # Dom
     .otherwise("entre_semana")                                      # Lun-Jue
)

# Columna combinada de estrato
df = df.withColumn(
    "estrato",
    F.concat_ws("_", "franja_horaria", "zona_geo", "tipo_dia")
)

print("Variables de particionamiento asignadas.")
df.select("tpep_pickup_datetime", "franja_horaria", "zona_geo", "tipo_dia", "estrato").show(10, truncate=False)

Variables de particionamiento asignadas.
+--------------------+--------------+---------+-------------+-----------------------------+
|tpep_pickup_datetime|franja_horaria|zona_geo |tipo_dia     |estrato                      |
+--------------------+--------------+---------+-------------+-----------------------------+
|2015-01-15 19:05:39 |noche         |manhattan|entre_semana |noche_manhattan_entre_semana |
|2015-01-10 20:33:38 |noche         |manhattan|fin_de_semana|noche_manhattan_fin_de_semana|
|2015-01-10 20:33:38 |noche         |manhattan|fin_de_semana|noche_manhattan_fin_de_semana|
|2015-01-10 20:33:39 |noche         |manhattan|fin_de_semana|noche_manhattan_fin_de_semana|
|2015-01-10 20:33:39 |noche         |manhattan|fin_de_semana|noche_manhattan_fin_de_semana|
|2015-01-10 20:33:39 |noche         |laguardia|fin_de_semana|noche_laguardia_fin_de_semana|
|2015-01-10 20:33:39 |noche         |manhattan|fin_de_semana|noche_manhattan_fin_de_semana|
|2015-01-10 20:33:39 |noche         |ma

## Muestreo

De los 46 millones de registros, tomaremos cerca de 50,000 para realizar el entrenamiento, buscando que mantengan la proporción por estratos. Los estratos más pequeños corren el riesgo de estar subrepresentados, por lo cual se establece un mínimo de 100 para cada muestra y después se asigna proporcionalmente las cantidades restantes.

In [12]:
from pyspark.sql.window import Window
import gc

N_TOTAL = n_limpio
N_TARGET = 50_000
N_MIN    = 100
H        = 48

# Paso 1: calcular fracciones de muestreo por estrato
budget = N_TARGET - N_MIN * H

strata_counts = df.groupBy("estrato").count().collect()
fractions = {
    row["estrato"]: (N_MIN + budget * row["count"] / N_TOTAL) / row["count"]
    for row in strata_counts
}
# Clampear a 1.0 en caso de estratos con pocos registros
fractions = {k: min(v, 1.0) for k, v in fractions.items()}

# Paso 2: muestreo estratificado
sample_df = df.sampleBy("estrato", fractions=fractions, seed=42)

# Paso 3: calcular pesos de muestra para corregir sesgo
n_sampled = sample_df.groupBy("estrato").count().withColumnRenamed("count", "n_h")
strata_df  = df.groupBy("estrato").count().withColumnRenamed("count", "N_h")

# Alias the dataframes before joining to avoid ambiguous column references
weights_df = strata_df.alias("s").join(n_sampled.alias("ns"), "estrato").withColumn(
    "sample_weight",
    (F.col("s.N_h") / N_TOTAL) / (F.col("ns.n_h") / N_TARGET)
)

# Paso 4: unir pesos al dataset de muestra
sample_df = sample_df.join(weights_df.select("estrato", "sample_weight"), "estrato")
sample_df = sample_df.cache()

# Borrar df de la memoria
#del df
#gc.collect()

In [9]:
## Verificar tamaño de la muestra
total = sample_df.count()
print(f"Tamaño de la muestra: {total:,}")


Tamaño de la muestra: 49,861


In [13]:
## Definir función para calcular distancia entre puntos geográficos (haversine)

def haversine_m(lat1, lon1, lat2, lon2):
    """
    Calcula la distancia haversine entre dos puntos geográficos.
    Devuelve una Column con la distancia en metros.
    """

    R = 6_371_000
    phi1    = F.radians(lat1)
    phi2    = F.radians(lat2)
    dphi    = F.radians(lat2 - lat1)
    dlambda = F.radians(lon2 - lon1)

    a = (F.pow(F.sin(dphi / 2), 2)
         + F.cos(phi1) * F.cos(phi2) * F.pow(F.sin(dlambda / 2), 2))
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))

    return F.lit(R) * c

In [37]:
## Añadir distancia calculada y tiempo calculado (target)
sample_df = sample_df.withColumn(
    "trip_distance_m",
    haversine_m(
        F.col("pickup_latitude"),
        F.col("pickup_longitude"),
        F.col("dropoff_latitude"),
        F.col("dropoff_longitude")
    )
)

sample_df = sample_df.withColumn(
    "trip_duration_min",
    (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60
)

# Filtrar outliers de duración y distancia
sample_df = sample_df.filter(
    (F.col("trip_duration_min") > 3) &
    (F.col("trip_duration_min") <= 180) &
    (F.col("trip_distance_m") > 50) &
    (F.col("trip_distance_m") <= 50_000)
)



In [38]:
## Mostrar df transformado

sample_df.show(5)

+--------------------+--------------------+---------------------+------------------+-----------------+------------------+------------------+--------------+-----------+--------------+--------+------------+------------------+------------------+------------------+
|             estrato|tpep_pickup_datetime|tpep_dropoff_datetime|  pickup_longitude|  pickup_latitude| dropoff_longitude|  dropoff_latitude|pickup_weekday|pickup_hour|franja_horaria|zona_geo|    tipo_dia|     sample_weight|   trip_distance_m| trip_duration_min|
+--------------------+--------------------+---------------------+------------------+-----------------+------------------+------------------+--------------+-----------+--------------+--------+------------+------------------+------------------+------------------+
|noche_outer_entre...| 2016-03-31 23:55:49|  2016-04-01 00:02:28| -73.9890365600586| 40.6939697265625|-73.99463653564453| 40.71682357788086|             5|         23|         noche|   outer|entre_semana|0.57694351

## Conjunto de entrenamiento y prueba

Para evitar sesgos o subrepresentación, dividiremos el conjunto de entrenamiento y de prueba de manera que se mantenga lo más posible la proporción entre estratos.

In [39]:
TRAIN_FRACTION = 0.8
SEED = 42

print("Dividir por estrato")
fracs_split = {
    row["estrato"]: TRAIN_FRACTION
    for row in sample_df.select("estrato").distinct().collect()
}

print("Crear particiones de entrenamiento y prueba")

df_train = sample_df.sampleBy("estrato", fractions=fracs_split, seed=SEED)
df_test  = sample_df.join(df_train, on=sample_df.columns, how="left_anti")

print("Crear cache")
df_train = df_train.cache()
df_test  = df_test.cache()

print("Calcular tamaño de particiones")

train_total = df_train.count()
test_total  = df_test.count()

total = train_total + test_total

print(f"Train: {train_total:,}")
print(f"Test:  {test_total:,}")

Dividir por estrato
Crear particiones de entrenamiento y prueba
Crear cache
Calcular tamaño de particiones
Train: 37,570
Test:  9,434


### Verificación

Verificamos que los estratos guarden proporción entre los conjuntos de entrenamiento y prueba.

In [40]:
print(f"Muestra : {total:,}")
print(f"Train   : {train_total:,}  ({train_total/total:.1%})")
print(f"Test    : {test_total:,}   ({test_total/total:.1%})")

# Proporciones por estrato
def stratum_props(df, label):
    n = df.count()
    return (
        df.groupBy("estrato")
          .count()
          .withColumn(f"pct_{label}", F.round(F.col("count") / n * 100, 2))
          .drop("count")
    )

(
    stratum_props(sample_df, "sample")
    .join(stratum_props(df_train, "train"), "estrato")
    .join(stratum_props(df_test,  "test"),  "estrato")
    .orderBy("estrato")
    .show(48, truncate=False)
)

Muestra : 47,004
Train   : 37,570  (79.9%)
Test    : 9,434   (20.1%)
+---------------------------------+----------+---------+--------+
|estrato                          |pct_sample|pct_train|pct_test|
+---------------------------------+----------+---------+--------+
|madrugada_jfk_entre_semana       |0.3       |0.29     |0.33    |
|madrugada_jfk_fin_de_semana      |0.29      |0.28     |0.31    |
|madrugada_jfk_valle              |0.23      |0.22     |0.29    |
|madrugada_laguardia_entre_semana |0.23      |0.23     |0.22    |
|madrugada_laguardia_fin_de_semana|0.24      |0.23     |0.24    |
|madrugada_laguardia_valle        |0.19      |0.19     |0.18    |
|madrugada_manhattan_entre_semana |3.38      |3.42     |3.24    |
|madrugada_manhattan_fin_de_semana|4.13      |4.13     |4.11    |
|madrugada_manhattan_valle        |2.81      |2.78     |2.93    |
|madrugada_outer_entre_semana     |0.3       |0.3      |0.28    |
|madrugada_outer_fin_de_semana    |0.35      |0.32     |0.43    |
|madrug

In [43]:
# ── Baseline: media por estrato (calculada SOLO sobre train) ────────────────
MIN_N_BASELINE = 100
SEED = 42

# Muestra reducida para el baseline
fracs = (
    df_train
    .groupBy("estrato")
    .count()
    .rdd
    .map(lambda row: (row["estrato"], min(1.0, MIN_N_BASELINE / row["count"])))
    .collectAsMap()
)

df_train_small = df_train.sampleBy("estrato", fractions=fracs, seed=SEED)

print("Imprimir media por estrato")

(
    df_train_small
    .groupBy("estrato")
    .agg(
        F.mean("trip_duration_min").alias("trip_duration_min_mean"),
        F.mean("trip_distance_m").alias("trip_distance_m_mean")
    )
    .orderBy("estrato")
    .show(48, truncate=False)
)

Imprimir media por estrato
+---------------------------------+----------------------+--------------------+
|estrato                          |trip_duration_min_mean|trip_distance_m_mean|
+---------------------------------+----------------------+--------------------+
|madrugada_jfk_entre_semana       |27.26666666666667     |16947.698283528516  |
|madrugada_jfk_fin_de_semana      |27.74579124579126     |17437.16756037448   |
|madrugada_jfk_valle              |25.17201646090535     |16293.946830950317  |
|madrugada_laguardia_entre_semana |18.917054263565888    |8817.961610178736   |
|madrugada_laguardia_fin_de_semana|19.692045454545454    |9608.381424270667   |
|madrugada_laguardia_valle        |17.78634259259259     |9117.396470209365   |
|madrugada_manhattan_entre_semana |11.944771241830066    |4433.545346528527   |
|madrugada_manhattan_fin_de_semana|12.67766666666667     |4440.048853250483   |
|madrugada_manhattan_valle        |12.237765957446808    |3555.7197371864872  |
|madrugada_ou

In [46]:
# Establecer valores bases de comparación
mean_by_estrato = (
    df_train_small
    .groupBy("estrato")
    .agg(F.mean("trip_duration_min").alias("prediction"))
)

df_baseline = df_test.join(mean_by_estrato, on="estrato", how="left")

evaluator_b = RegressionEvaluator(
    labelCol="trip_duration_min",
    predictionCol="prediction"
)

print("── Baseline (media por estrato) ──")
for metric in ["rmse", "mae", "r2"]:
    evaluator_b.setMetricName(metric)
    print(f"  {metric.upper():>4}: {evaluator_b.evaluate(df_baseline):.4f}")

── Baseline (media por estrato) ──
  RMSE: 9.1915
   MAE: 6.5592
    R2: 0.3147


In [41]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# ── 1. Encoding de categóricas ──────────────────────────────────────────────
print("Encoding de variables categóricas")
indexers = [
    StringIndexer(inputCol="franja_horaria", outputCol="franja_idx"),
    StringIndexer(inputCol="tipo_dia",       outputCol="tipo_dia_idx"),
]

encoders = [
    OneHotEncoder(inputCol="franja_idx",   outputCol="franja_ohe"),
    OneHotEncoder(inputCol="tipo_dia_idx", outputCol="tipo_dia_ohe"),
]

# ── 2. Ensamble de features ─────────────────────────────────────────────────
print("Ensamble de features")
feature_cols = [
    "trip_distance_m",
    "pickup_hour",
    "pickup_weekday",
    "pickup_latitude",
    "pickup_longitude",
    "dropoff_latitude",
    "dropoff_longitude",
    "franja_ohe",
    "tipo_dia_ohe",
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

# ── 3. Escalado (se fitea solo sobre train) ─────────────────────────────────
print("Escalado")
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

# ── 4. Modelo ───────────────────────────────────────────────────────────────
print("Creación de Modelo de regresión lineal")
lr = LinearRegression(
    featuresCol="features",
    labelCol="trip_duration_min",
    weightCol="sample_weight",   # usa los pesos del muestreo estratificado
    maxIter=100,
    regParam=0.01,               # L2 ligero para estabilidad
    elasticNetParam=0.0
)

# ── 5. Pipeline ─────────────────────────────────────────────────────────────
print("Correr pipeline")

pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, lr])

pipeline_model = pipeline.fit(df_train)

Encoding de variables categóricas
Ensamble de features
Escalado
Creación de Modelo de regresión lineal
Correr pipeline


## Evaluación

In [42]:
df_pred = pipeline_model.transform(df_test)

evaluator = RegressionEvaluator(
    labelCol="trip_duration_min",
    predictionCol="prediction"
)

print("── Modelo (regresión lineal) ──")
for metric in ["rmse", "mae", "r2"]:
    evaluator.setMetricName(metric)
    print(f"  {metric.upper():>4}: {evaluator.evaluate(df_pred):.4f}")

── Modelo (regresión lineal) ──
  RMSE: 6.6645
   MAE: 4.4761
    R2: 0.6397


### Conclusiones

El error medio del modelo es mejor que el establecido como _baseline_ tomando los valores medios de cada estrato, mientras que nuestro modelo es capaz de explicar el 63% de la varianza.

Hay algunas limitaciones con nuestro método, ya que consideramos la distancia de punto A a B, sin que sea esta distancia realmente represente la mejor ruta en Nueva York.

Considerando esto y el hecho de que apilamos las variables de hora, día y zona, probablemente un modelo de árbol de decisión podría ofrecernos una mejor aproximación mejor que regresión lineal, pero requeriría también una muestra mayor de cada estrato para evitar _overfitting_.

### Modelos del lenguaje utilizados

- Anthropic. (2026). Claude Sonnet 4.6 [Modelo de lenguaje grande], utilizado para generación de código y depuración.
- Google. (2026). Gemini Flash 2.5 [Modelo de lenguaje grande], utilizado para generación de código y depuración.